# Decision Tree Visualisation

In [ ]:
# Model Persistence
from joblib import load
from sklearn.tree import export_graphviz
import graphviz

In [ ]:
import os
from pathlib import Path

TREE_FOLDER = Path(os.environ["DATA_BASE_DIR"]) / "trees"

os.makedirs(TREE_FOLDER, exist_ok=True)

Decision Tree (w/ Under Sampling, Near Miss)

In [ ]:
# Export the decision tree to a dot file
def visualize_tree(dt, feature_names: list[str], label: str, root_folder: str = TREE_FOLDER, export: bool = True):
    dot_data = export_graphviz(
        dt, out_file=None, feature_names=feature_names, class_names=["No Fraud", "Fraud"], filled=True
    )
    graph = graphviz.Source(dot_data)
    if export:
        label = label.lower().replace(" ", "_")
        # Save the graph as a jpg or pdf file
        graph.format = "jpg"  #'pdf'
        graph.render(root_folder / label)

    return graph

In [ ]:
import os

dt_near_miss = os.getenv("DECISION_TREE_NMISS")
rf_near_miss = os.getenv("RANDOM_FOREST_NMISS")

dt_smote = os.getenv("DECISION_TREE_SMOTE")
rf_smote = os.getenv("RANDOM_FOREST_SMOTE")

In [ ]:
gs_dt_nearmiss = load(dt_near_miss)

dt = gs_dt_nearmiss.best_estimator_.named_steps["model"]
feature_names = gs_dt_nearmiss.feature_names_in_

visualize_tree(dt, feature_names=feature_names, label="Decision Tree Near Miss", export=True)

In [ ]:
gs_dt_smote = load(dt_smote)

dt_smote = gs_dt_smote.best_estimator_.named_steps["model"]
feature_names_smote = gs_dt_smote.feature_names_in_

visualize_tree(dt_smote, feature_names=feature_names_smote, label="Decision Tree SMOTE", export=True)

## Random Forest

In [ ]:
gs_rf_nm = load(rf_near_miss)

rf_nm = gs_rf_nm.best_estimator_.named_steps["model"]
feature_names_nm = gs_rf_nm.feature_names_in_

rf_nm_root_folder = TREE_FOLDER / "rf_near_miss"
os.makedirs(rf_nm_root_folder, exist_ok=True)

for t, tree in enumerate(rf_nm.estimators_):
    _ = visualize_tree(
        tree, feature_names=feature_names_smote, label=f"Random Forest NearMiss (DT{t})", root_folder=rf_nm_root_folder
    )

In [ ]:
gs_rf_smote = load(rf_smote)

rf_smote = gs_rf_smote.best_estimator_.named_steps["model"]
feature_names_smote = gs_rf_smote.feature_names_in_

rf_smote_root_folder = TREE_FOLDER / "rf_smote"
os.makedirs(rf_smote_root_folder, exist_ok=True)

for t, tree in enumerate(rf_smote.estimators_):
    _ = visualize_tree(
        tree, feature_names=feature_names_smote, label=f"Random Forest SMOTE (DT{t})", root_folder=rf_smote_root_folder
    )